# HNSCC Case Study + Double Circos Visualization

This notebook shows how to visualize the output of a NicheNet analysis in a circos plot. Here, we use a normal expression matrix as input instead of an AnnData/Seurat object.

We use the same HNSCC dataset as the ligand activity geneset notebook (Puram et al., 2017). We predict which ligands expressed by both CAFs and endothelial cells can induce the p-EMT program in neighboring malignant cells.

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nichenetr as nn

### Load networks and expression data

In [ ]:
lr_network = nn.load_lr_network("human")
ligand_target_matrix = nn.load_ligand_target_matrix("human")
weighted_networks = nn.load_weighted_networks("human")

lr_network = lr_network[["from", "to"]].drop_duplicates()

In [ ]:
hnscc_data = nn.load_hnscc_expression()
expression = hnscc_data["expression"]
sample_info = hnscc_data["sample_info"]

expression_df = pd.DataFrame(
    expression.data.toarray(),
    index=expression.rownames,
    columns=expression.colnames,
)

# Convert aliases
expression_df.columns = nn.convert_alias_to_symbols(
    list(expression_df.columns), "human", verbose=False
)

### Define a set of potential ligands

In [ ]:
tumors_remove = ["HN10", "HN", "HN12", "HN13", "HN24", "HN7", "HN8", "HN23"]

CAF_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["non-cancer cell type"] == "CAF")
]["cell"].tolist()

endothelial_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["non-cancer cell type"] == "Endothelial")
]["cell"].tolist()

malignant_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["classified  as cancer cell"] == 1)
]["cell"].tolist()

def get_expressed_custom(expression_df, cell_ids):
    sub = expression_df.loc[expression_df.index.isin(cell_ids)]
    tpm = 10 * (2 ** sub - 1)
    agg = np.log2(tpm.mean(axis=0) + 1)
    return agg[agg >= 4].index.tolist()

expressed_genes_CAFs = get_expressed_custom(expression_df, CAF_ids)
expressed_genes_endothelial = get_expressed_custom(expression_df, endothelial_ids)
expressed_genes_malignant = get_expressed_custom(expression_df, malignant_ids)

ligands = lr_network["from"].unique().tolist()
expressed_ligands_CAFs = list(set(ligands) & set(expressed_genes_CAFs))
expressed_ligands_endothelial = list(set(ligands) & set(expressed_genes_endothelial))
expressed_ligands = list(set(expressed_ligands_CAFs) | set(expressed_ligands_endothelial))

receptors = lr_network["to"].unique().tolist()
expressed_receptors = list(set(receptors) & set(expressed_genes_malignant))

potential_ligands = (
    lr_network[
        lr_network["from"].isin(expressed_ligands)
        & lr_network["to"].isin(expressed_receptors)
    ]["from"].unique().tolist()
)

### Define the gene set and background

In [ ]:
pemt_geneset = nn.load_pemt_signature()
pemt_geneset = [g for g in pemt_geneset if g in ligand_target_matrix.rownames]

background_expressed_genes = [
    g for g in expressed_genes_malignant if g in ligand_target_matrix.rownames
]

### Perform NicheNet's ligand activity analysis

In [ ]:
ligand_activities = nn.predict_ligand_activities(
    geneset=pemt_geneset,
    background_expressed_genes=background_expressed_genes,
    ligand_target_matrix=ligand_target_matrix,
    potential_ligands=potential_ligands,
)

# Top 20 ligands
best_upstream_ligands = (
    ligand_activities.nlargest(20, "aupr_corrected")["test_ligand"].tolist()
)
print(best_upstream_ligands)

### Determine ligand cell-type specificity

Determine whether each ligand is more strongly expressed in CAFs or endothelial cells.

In [ ]:
# Average expression per cell type for top ligands
available_ligands = [l for l in best_upstream_ligands if l in expression_df.columns]

caf_avg = expression_df.loc[expression_df.index.isin(CAF_ids), available_ligands]
caf_tpm = 10 * (2 ** caf_avg - 1)
caf_expr = np.log2(caf_tpm.mean(axis=0) + 1)

endo_avg = expression_df.loc[expression_df.index.isin(endothelial_ids), available_ligands]
endo_tpm = 10 * (2 ** endo_avg - 1)
endo_expr = np.log2(endo_tpm.mean(axis=0) + 1)

ligand_expression_tbl = pd.DataFrame({
    "ligand": available_ligands,
    "CAF": caf_expr.values,
    "endothelial": endo_expr.values,
})

CAF_specific = ligand_expression_tbl[
    ligand_expression_tbl["CAF"] > ligand_expression_tbl["endothelial"] + 2
]["ligand"].tolist()
endo_specific = ligand_expression_tbl[
    ligand_expression_tbl["endothelial"] > ligand_expression_tbl["CAF"] + 2
]["ligand"].tolist()
general = [l for l in available_ligands if l not in CAF_specific and l not in endo_specific]

ligand_type_indication_df = pd.DataFrame({
    "ligand_type": (
        ["CAF-specific"] * len(CAF_specific)
        + ["General"] * len(general)
        + ["Endothelial-specific"] * len(endo_specific)
    ),
    "ligand": CAF_specific + general + endo_specific,
})

ligand_type_indication_df

### Infer target genes and visualize with circos plot

In [ ]:
active_ligand_target_links_df = pd.concat(
    [
        nn.get_weighted_ligand_target_links(
            ligand_oi=lig,
            geneset=pemt_geneset,
            ligand_target_matrix=ligand_target_matrix,
            n=250,
        )
        for lig in best_upstream_ligands
    ],
    ignore_index=True,
).dropna()

active_ligand_target_links_df["target_type"] = "p_emt"

In [ ]:
# Filter links with weight cutoff
circos_links = nn.get_ligand_target_links_oi(
    ligand_type_indication_df,
    active_ligand_target_links_df,
    cutoff=0.66,
)

In [ ]:
ligand_colors = {
    "General": "lawngreen",
    "CAF-specific": "royalblue",
    "Endothelial-specific": "gold",
}
target_colors = {"p_emt": "tomato"}

vis_circos_obj = nn.prepare_circos_visualization(
    circos_links,
    ligand_colors=ligand_colors,
    target_colors=target_colors,
)

In [ ]:
# Circos plot without transparency
nn.make_circos_plot(vis_circos_obj, transparency=False, show=True)

In [ ]:
# Circos plot with transparency
nn.make_circos_plot(vis_circos_obj, transparency=True, show=True)

### Visualize ligand-receptor interactions as a circos plot

In [ ]:
lr_network_top_df = nn.get_weighted_ligand_receptor_links(
    best_upstream_ligands,
    expressed_receptors,
    lr_network,
    weighted_networks["lr_sig"],
)

lr_network_top_df = lr_network_top_df.rename(columns={"from": "ligand", "to": "target"})
lr_network_top_df["target_type"] = "p_emt_receptor"
lr_network_top_df = lr_network_top_df.merge(ligand_type_indication_df, on="ligand")

receptor_colors = {"p_emt_receptor": "darkred"}

vis_circos_receptor_obj = nn.prepare_circos_visualization(
    lr_network_top_df,
    ligand_colors=ligand_colors,
    target_colors=receptor_colors,
)

In [ ]:
# Ligand-receptor circos plot
nn.make_circos_plot(
    vis_circos_receptor_obj,
    transparency=False,
    link_visible=True,
    show=True,
)

In [ ]:
# With transparency
nn.make_circos_plot(
    vis_circos_receptor_obj,
    transparency=True,
    link_visible=True,
    show=True,
)

### Double circos plot (ligand-receptor-target)

A "ligand-receptor-target" circos plot can be created by making two separate circos plots (ligand-target and ligand-receptor) and overlaying them. In the R version this is done via `draw.sector` or `highlight.sector` from `circlize`. In Python, you can achieve similar effects by manually drawing arcs on a matplotlib figure or using pycirclize. Below is a simplified demonstration grouping targets by ligand groups.

In [ ]:
# Define ligand groups for the double circos (subset of top ligands)
groups = {
    "group1": [l for l in best_upstream_ligands if l in ["TGFB2", "ENG"] and l in active_ligand_target_links_df["ligand"].values],
    "group2": [l for l in best_upstream_ligands if any(x in l for x in ["BMP", "GDF", "INHBA"]) and l in active_ligand_target_links_df["ligand"].values],
    "group3": [l for l in best_upstream_ligands if any(x in l for x in ["COL", "MMP", "TIMP"]) and l in active_ligand_target_links_df["ligand"].values],
    "group4": [l for l in best_upstream_ligands if l == "CXCL12" and l in active_ligand_target_links_df["ligand"].values],
}

# Remove empty groups
groups = {k: v for k, v in groups.items() if len(v) > 0}

print("Ligand groups:")
for k, v in groups.items():
    print(f"  {k}: {v}")

In [ ]:
# For each group, find the top targets
all_group_ligands = [l for group in groups.values() for l in group]

subset_links = active_ligand_target_links_df[
    active_ligand_target_links_df["ligand"].isin(all_group_ligands)
].copy()

if len(subset_links) > 0:
    # Assign targets to groups based on strongest weight
    target_assignments = []
    for group_name, group_ligands in groups.items():
        group_links = subset_links[subset_links["ligand"].isin(group_ligands)]
        for target, grp in group_links.groupby("target"):
            target_assignments.append({
                "target": target,
                "group": group_name,
                "weight": grp["weight"].sum(),
            })
    
    target_df = pd.DataFrame(target_assignments)
    if len(target_df) > 0:
        # Assign each target to its highest-weight group
        target_df = target_df.sort_values("weight", ascending=False).drop_duplicates("target")
        print(f"Targets assigned to groups: {len(target_df)}")
        print(target_df["group"].value_counts())

For a full double circos plot implementation, you can render the inner (ligand-target) circos and outer (ligand-receptor) ring separately, then combine them. The exact approach depends on whether you use pycirclize or custom matplotlib arcs. See the R vignette for the conceptual workflow, and adapt using pycirclize's sector highlighting capabilities.